[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Metodos_Matematicos_%28MTH%29/Simulacion_y_Machine_Learning_%28physics.comp-ph%29/MTH-07_PINN_Descubrimiento_Ecuaciones_Pendulo.ipynb)

# [MTH-07] Descubrimiento de Leyes (Problema Inverso)

Nota: Este emulador utiliza `scipy.optimize` y derivadas simbólicas en NumPy para ilustrar el mecanismo subyacente de Autograd (PyTorch/JAX) en un entorno puramente didáctico de PINNs sin dependencias de tensores GPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# PINN Equation Discovery (Problema Inverso)
# Datos ruidosos de un Péndulo: u_tt + lambda_1 * u_t + lambda_2 * sin(u) = 0
# La PINN no conoce lambda_1 (fricción) ni lambda_2 (gravedad g/L). Debe descubrirlos.

# 1. Generamos "Datos Reales" (Ocultos)
np.random.seed(0)
t_data = np.linspace(0, 10, 50)
true_l1, true_l2 = 0.5, 9.81
# Aprox u(t) = exp(-l1*t/2) * cos(sqrt(l2)*t)
u_real = np.exp(-true_l1 * t_data / 2.0) * np.cos(np.sqrt(true_l2) * t_data)
# Contaminamos con ruido
u_noisy = u_real + 0.1 * np.random.randn(len(t_data))

# 2. Arquitectura PINN + Parámetros Inversos
# params: [w1, w2, b1, b2, lambda_1_pred, lambda_2_pred]
def pinn_loss(params):
    w1, w2, b1, b2, l1_pred, l2_pred = params
    
    # Forward pass: u(t) = w2 * sin(w1 * t + b1) + b2
    u_pred = w2 * np.sin(w1 * t_data + b1) + b2
    
    # Derivadas
    u_t = w2 * w1 * np.cos(w1 * t_data + b1)
    u_tt = -w2 * w1**2 * np.sin(w1 * t_data + b1)
    
    # 1. Data Loss (Ajuste a los datos observados)
    loss_data = np.mean((u_pred - u_noisy)**2)
    
    # 2. Physics Loss (Gobernado por los lambdas descubiertos por la red)
    f_res = u_tt + l1_pred * u_t + l2_pred * np.sin(u_pred)
    loss_phys = np.mean(f_res**2)
    
    return loss_data + loss_phys

# Inicializamos la red con parámetros aleatorios y lambdas falsos (1.0, 1.0)
init_p = [1.0, 1.0, 0.0, 0.0, 1.0, 1.0]
res = minimize(pinn_loss, init_p, method='BFGS')

discovered_l1 = res.x[4]
discovered_l2 = res.x[5]

print(f"Fricción Real (λ1): {true_l1} | Fricción Descubierta: {discovered_l1:.4f}")
print(f"Gravedad Real (λ2): {true_l2} | Gravedad Descubierta: {discovered_l2:.4f}")

plt.figure(figsize=(10,5))
plt.plot(t_data, u_noisy, 'ko', label="Datos de Sensores (Ruidosos)")
plt.plot(t_data, res.x[1] * np.sin(res.x[0] * t_data + res.x[2]) + res.x[3], 'r-', lw=3, label="Curva Aprendida por PINN")
plt.title("Descubrimiento de Ecuaciones: PINN Deduciendo Gravedad")
plt.xlabel("Tiempo")
plt.ylabel("Ángulo de Péndulo")
plt.legend()
plt.grid()
plt.show()

